# MSI Pipeline - Script 04: Spatial Analysis

This notebook computes spatial statistics for MSI data, primarily Moran's I for spatial autocorrelation.

## Features
- **Moran's I calculation** per channel using `esda.moran.Moran`
- KNN weights construction (k=6, configurable)
- Spatial autocorrelation filtering by modality thresholds

## Modality Thresholds (Moran's I)
- Glycans: > 0.2
- Metabolites: > 0.05
- Peptides: > 0.2

## Input
- Preprocessed AnnData files from script03

## Output
- Moran's I results per channel (CSV)
- List of spatially coherent channels
- Updated AnnData with spatial statistics

In [ ]:
import sys
from pathlib import Path
import logging
import warnings

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))

from utils import spatial as msi_spatial
from utils import io as msi_io

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Suppress libpysal warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Work around matplotlib_inline expecting rcParams._get in some versions
if not hasattr(mpl.rcParams, "_get"):
    mpl.rcParams._get = mpl.rcParams.__getitem__

# Plot settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

## Configuration

In [ ]:
# === CONFIGURE THESE PATHS ===

# Base data directory
BASE_DIR = Path(r"T:/Sammy Data/Third set results/")  # Path.home() / "ext_hd_sammy"
# BASE_DIR = Path.home() / "ext_hd_sammy"

# Input: Preprocessed AnnData files from script03
INPUT_DIR = BASE_DIR /  "peptides_h5ad_processed"

# Output directories
OUTPUT_DIR = BASE_DIR / "peptides_spatial_stats"

# Data modality (determines Moran's I threshold)
MODALITY = "peptides"  # Options: glycans, metabolites, peptides

# Spatial analysis parameters
KNN_K = 6  # Number of nearest neighbors for spatial weights
N_PERMUTATIONS = 999  # Number of permutations for significance testing
P_VALUE_THRESHOLD = 0.05  # Maximum p-value for significance

# Get modality-specific threshold
MORANS_I_THRESHOLD = msi_spatial.get_modality_threshold(MODALITY)

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Modality: {MODALITY}")
print(f"Moran's I threshold: {MORANS_I_THRESHOLD}")
print(f"KNN k: {KNN_K}")

## Discover Samples

In [ ]:
# Find all h5ad files
sample_files = list(INPUT_DIR.glob("*.h5ad"))
sample_ids = [f.stem for f in sample_files]

print(f"Found {len(sample_ids)} samples:")
for sid in sample_ids:
    print(f"  - {sid}")

## Load Sample

In [ ]:
# Load first sample
if sample_files:
    sample_file = sample_files[0]
    sample_id = sample_file.stem
    
    print(f"Loading {sample_id}...")
    adata = ad.read_h5ad(sample_file)
    
    print(f"Shape: {adata.shape}")
    print(f"Has spatial coordinates: {'spatial' in adata.obsm}")

## Subsample for Large Datasets (Optional)

In [ ]:
# For large datasets, subsample to speed up computation
MAX_PIXELS = 50000  # Set to None to use all pixels

if MAX_PIXELS and adata.n_obs > MAX_PIXELS:
    print(f"Subsampling from {adata.n_obs} to {MAX_PIXELS} pixels...")
    np.random.seed(42)
    idx = np.random.choice(adata.n_obs, MAX_PIXELS, replace=False)
    adata_subset = adata[idx, :].copy()
    print(f"Subsampled shape: {adata_subset.shape}")
else:
    adata_subset = adata

## Compute Moran's I

In [ ]:
# Get coordinates and data
coordinates = adata_subset.obsm['spatial']
X = adata_subset.X if not hasattr(adata_subset.X, 'toarray') else adata_subset.X.toarray()

print(f"Computing Moran's I for {adata_subset.n_vars} channels...")
print(f"Using {len(coordinates)} pixels with k={KNN_K} neighbors")
print(f"This may take several minutes...")

In [ ]:
# Compute Moran's I for all channels
morans_results = msi_spatial.morans_i_per_channel(
    data=X,
    coordinates=coordinates,
    k=KNN_K,
    permutations=N_PERMUTATIONS,
    channel_names=list(adata_subset.var_names),
)

print(f"\nMoran's I computed for {len(morans_results)} channels")
print(morans_results.head(10))

## Visualize Moran's I Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of Moran's I
ax = axes[0]
ax.hist(morans_results['I'].dropna(), bins=30, edgecolor='black', alpha=0.7)
ax.axvline(MORANS_I_THRESHOLD, color='r', linestyle='--', 
           label=f'Threshold: {MORANS_I_THRESHOLD}')
ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
ax.set_xlabel("Moran's I")
ax.set_ylabel('Count')
ax.set_title("Distribution of Moran's I")
ax.legend()

# Moran's I vs p-value
ax = axes[1]
ax.scatter(morans_results['I'], -np.log10(morans_results['p_value'] + 1e-10), 
           alpha=0.6)
ax.axhline(-np.log10(P_VALUE_THRESHOLD), color='r', linestyle='--', 
           label=f'p={P_VALUE_THRESHOLD}')
ax.axvline(MORANS_I_THRESHOLD, color='orange', linestyle='--', 
           label=f'I={MORANS_I_THRESHOLD}')
ax.set_xlabel("Moran's I")
ax.set_ylabel('-log10(p-value)')
ax.set_title("Moran's I vs Significance")
ax.legend()

# Top channels by Moran's I
ax = axes[2]
top_n = 20
top_channels = morans_results.nlargest(top_n, 'I')
ax.barh(range(len(top_channels)), top_channels['I'].values, alpha=0.7)
ax.set_yticks(range(len(top_channels)))
ax.set_yticklabels(top_channels['channel'].values)
ax.axvline(MORANS_I_THRESHOLD, color='r', linestyle='--')
ax.set_xlabel("Moran's I")
ax.set_title(f'Top {top_n} Channels by Spatial Autocorrelation')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{sample_id}_morans_i_summary.png", dpi=150, bbox_inches='tight')
plt.show()

## Filter Spatially Coherent Channels

In [ ]:
# Filter by Moran's I threshold and significance
coherent_channels = msi_spatial.filter_spatially_coherent(
    morans_results,
    threshold=MORANS_I_THRESHOLD,
    p_value_threshold=P_VALUE_THRESHOLD
)

print(f"\nSpatially coherent channels: {len(coherent_channels)}/{len(morans_results)}")
print(f"\nTop coherent channels:")
print(coherent_channels.head(10))

In [ ]:
# Save coherent channel list
coherent_channel_names = coherent_channels['channel'].tolist()

# Save as text file for easy reference
coherent_path = OUTPUT_DIR / f"{sample_id}_coherent_channels.txt"
with open(coherent_path, 'w') as f:
    f.write(f"# Spatially coherent channels for {sample_id}\n")
    f.write(f"# Modality: {MODALITY}\n")
    f.write(f"# Moran's I threshold: {MORANS_I_THRESHOLD}\n")
    f.write(f"# p-value threshold: {P_VALUE_THRESHOLD}\n")
    f.write(f"# Total: {len(coherent_channel_names)} channels\n\n")
    for name in coherent_channel_names:
        f.write(f"{name}\n")

print(f"Saved coherent channel list to {coherent_path}")

## Visualize Top Spatially Coherent Channels

In [ ]:
# Plot spatial distribution of top coherent channels
n_plot = min(6, len(coherent_channels))

if n_plot > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    coords = adata_subset.obsm['spatial']
    
    for i, (_, row) in enumerate(coherent_channels.head(n_plot).iterrows()):
        channel = row['channel']
        morans_i = row['I']
        
        # Get channel index
        ch_idx = list(adata_subset.var_names).index(channel)
        values = X[:, ch_idx]
        
        ax = axes[i]
        scatter = ax.scatter(coords[:, 0], coords[:, 1], c=values, 
                            s=1, cmap='viridis', alpha=0.5)
        plt.colorbar(scatter, ax=ax)
        ax.set_title(f"{channel}\nMoran's I = {morans_i:.3f}")
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
    
    # Hide unused axes
    for i in range(n_plot, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{sample_id}_top_coherent_spatial.png", 
                dpi=150, bbox_inches='tight')
    plt.show()

## Save Results

In [ ]:
# Save full Moran's I results
morans_path = OUTPUT_DIR / f"{sample_id}_morans_i.csv"
morans_results.to_csv(morans_path, index=False)
print(f"Saved Moran's I results to {morans_path}")

# Save filtered results
coherent_path = OUTPUT_DIR / f"{sample_id}_morans_i_coherent.csv"
coherent_channels.to_csv(coherent_path, index=False)
print(f"Saved coherent channels to {coherent_path}")

# Add Moran's I to AnnData var
morans_df = morans_results.set_index('channel')
adata.var['morans_i_I'] = morans_df.loc[adata.var_names, 'I'].values
adata.var['morans_i_p'] = morans_df.loc[adata.var_names, 'p_value'].values
adata.var['spatially_coherent'] = adata.var_names.isin(coherent_channel_names)

# Save updated AnnData
adata.write_h5ad(sample_file, compression='gzip')
print(f"Updated AnnData saved")

## Batch Process All Samples

In [ ]:
def process_sample_spatial(input_path, output_dir, modality, knn_k, 
                           n_permutations, p_threshold, max_pixels=50000):
    """Process a single sample for spatial statistics."""
    sample_id = input_path.stem
    logger.info(f"Processing {sample_id}...")
    
    # Load
    adata = ad.read_h5ad(input_path)
    
    # Subsample if needed
    if max_pixels and adata.n_obs > max_pixels:
        np.random.seed(42)
        idx = np.random.choice(adata.n_obs, max_pixels, replace=False)
        adata_subset = adata[idx, :].copy()
    else:
        adata_subset = adata
    
    # Get data
    coordinates = adata_subset.obsm['spatial']
    X = adata_subset.X if not hasattr(adata_subset.X, 'toarray') else adata_subset.X.toarray()
    
    # Compute Moran's I
    morans_results = msi_spatial.morans_i_per_channel(
        data=X,
        coordinates=coordinates,
        k=knn_k,
        permutations=n_permutations,
        channel_names=list(adata_subset.var_names),
    )
    
    # Get threshold and filter
    threshold = msi_spatial.get_modality_threshold(modality)
    coherent = msi_spatial.filter_spatially_coherent(
        morans_results, threshold=threshold, p_value_threshold=p_threshold
    )
    
    # Save
    morans_results.to_csv(output_dir / f"{sample_id}_morans_i.csv", index=False)
    coherent.to_csv(output_dir / f"{sample_id}_morans_i_coherent.csv", index=False)
    
    # Update AnnData
    morans_df = morans_results.set_index('channel')
    adata.var['morans_i_I'] = morans_df.loc[adata.var_names, 'I'].values
    adata.var['morans_i_p'] = morans_df.loc[adata.var_names, 'p_value'].values
    adata.var['spatially_coherent'] = adata.var_names.isin(coherent['channel'].tolist())
    adata.write_h5ad(input_path, compression='gzip')
    
    return {
        'sample_id': sample_id,
        'n_channels': len(morans_results),
        'n_coherent': len(coherent),
        'mean_morans_i': morans_results['I'].mean(),
        'status': 'success'
    }

In [ ]:
# Process all samples
all_results = []

for sample_file in sample_files:
    try:
        result = process_sample_spatial(
            input_path=sample_file,
            output_dir=OUTPUT_DIR,
            modality=MODALITY,
            knn_k=KNN_K,
            n_permutations=N_PERMUTATIONS,
            p_threshold=P_VALUE_THRESHOLD,
            max_pixels=MAX_PIXELS
        )
        all_results.append(result)
    except Exception as e:
        logger.error(f"Error processing {sample_file.stem}: {e}")
        all_results.append({
            'sample_id': sample_file.stem,
            'status': 'error',
            'error': str(e)
        })

# Summary
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(OUTPUT_DIR / "spatial_analysis_summary.csv", index=False)
print("\nSpatial Analysis Summary:")
print(summary_df)